In [1]:
import grad_exp as g
import pickle
import matplotlib.pyplot as plt
import numpy as np

from tqdm import tqdm 
from sklearn.datasets import fetch_openml
from sklearn.manifold import TSNE     #make sure you are using the editable version of sklearn
from sklearn.preprocessing import StandardScaler
from collections import Counter

In [3]:
### MNIST
X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)
X_scaled=X/255.0
train=X_scaled[:1000]
labels_train=y[:1000]
test=X_scaled[1000:1500]
labels_test=y[1000:1500]

In [4]:
tsne=TSNE(n_components=2,
          perplexity=15,
          random_state=42,
          method="exact")
feat=tsne.fit_transform(train)
#if correctly using editable version of sklearn (as stated in the github) then it should print "here"

here


In [ ]:
d=train.shape[1]
ge_values_list_l2=[np.zeros((d))]*len(set(labels_train))

### by labels
#very time expensive
for idx, i in tqdm(enumerate(labels_train[:50])):
    direct_2order=g.calc_direct_2order(tsne, feat, idx)
    cross_2order_tot=g.cross_2order_total(tsne, train, feat, idx)
    grad_x=g.grad_wtr_x(direct_2order, cross_2order_tot)

    l2=np.sqrt(np.square(grad_x[0]) + np.square(grad_x[1]))
    ge_values_list_l2[int(i)]+=l2

counts=Counter(labels_train)
for i in set(labels_train):
    ge_values_list_l2[int(i)]/=counts[i]

0it [00:00, ?it/s]

In [13]:
# overlaying the features above on original data image
def overlay_features(fa_values_list, title, top_k=10):
     '''
    This function colors the top k pixels with the highest magnitude of 
    feature attribution values on top of the grayscaled image for each digit class. 
    This function shows local attribution.
    
    Input: 
        fa_values: a list of 10 lists of local feature attribution values for each digit class 
        title: str; title for the figure
        top_k: int; how many pixels to color
    Output:
        Displays a 2 by 5 plot of gray scaled digits 0-9 with colored pixels.
    '''
    
    fig, ax=plt.subplots(2,5,figsize=(10,5))
    for i in range(10):   #from digits 0-9
        fa_val=fa_values_list[i][0]  #on the first image 
        og_image=train[labels_train==str(i)][0].reshape(28,28)  #on the first image

        top_indices=np.argsort(np.abs(fa_val), axis=None)[-top_k:]
        top_y,top_x=np.unravel_index(top_indices,(28,28))

        colors=np.zeros((28,28,3), dtype=float)
        for y,x in zip(top_y,top_x):
            if fa_val[y*28+x].item()>0:
                colors[y,x]=[1,0,0]  #red if attribution value is positive
            else:
                colors[y,x]=[0,0,1]  #blue if attribution value is negative
        ax[i//5, i%5].imshow(og_image, cmap="gray")
        ax[i//5, i%5].imshow(colors, alpha=0.6)
        ax[i//5, i%5].axis("off")
    fig.suptitle(title)

def global_overlay_features(fa_values_avg, title, top_k=10):
    '''
    This function colors the top k pixels with the highest magnitude of 
    feature attribution values on top of the averaged grayscaled image of each digit class. 
    This function shows global attribution.
    
    Input: 
        fa_avg_values: a list of 10 lists of averaged feature attribution values for each digit class 
        title: str; title for the figure
        top_k: int; how many pixels to color
    Output:
        Displays a 5 by 2 plot of gray scaled digits 0-9 with colored pixels.
    '''
    
    fig, ax=plt.subplots(2,5,figsize=(10,5))
    for i in range(10):
        fa_val=fa_values_avg[i]
        og_image=train[labels_train==str(i)].mean(axis=0).reshape(28,28)

        top_indices=np.argsort(np.abs(fa_val), axis=None)[-top_k:]
        top_y,top_x=np.unravel_index(top_indices,(28,28))

        colors=np.zeros((28,28,3), dtype=float)
        for y,x in zip(top_y,top_x):
            if ig_val[y*28+x].item()>0:
                colors[y,x]=[1,0,0]  #red if avg attr value is positive
            else:
                colors[y,x]=[0,0,1]  #blue if avg attr value is negative
        ax[i//5, i%5].imshow(og_image, cmap="gray")
        ax[i//5, i%5].imshow(colors, alpha=0.6)
        ax[i//5, i%5].axis("off")
    fig.suptitle(title)

def heatmap_features(fa_values_avg, title):
    '''
    This function plots a heatmap of the global feature attribution values
    Input: 
        fa_values: a list of 10 lists of averaged feature attribution values for each digit class
        title: str; title for the figure
    Output:
        Displays a 5 by 2 heatmap plot of global feature attribution values.
    '''
    
    fig, ax=plt.subplots(2,5,figsize=(10,5))
    for i in range(10):
        fa_val=fa_values_avg[i]/np.sum(fa_values_avg[i])
        
        ax[i//5, i%5].imshow(fa_val.reshape(28,28),cmap="turbo")
        ax[i//5, i%5].axis("off")
    fig.suptitle(title)

In [ ]:
global_overlay_features(ge_values_list_l2, "grad exp l2", top_k=30)
heatmap_features(ge_values_list_l2, "grad exp l2")